### RAG Pipeline -> Data ingestion tp Vector dB Pipeline

In [10]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [14]:
import os
from pathlib import Path

from langchain_community.document_loaders import PyMuPDFLoader


### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Processes all PDF files in the specified directory and returns a list of documents."""

    all_documents = []

    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.rglob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in the directory '{pdf_directory}'.")

    for pdf_file in pdf_files:

        print(f"\nProcessing file: {pdf_file}")

        try:
            # Load PDF
            loader = PyMuPDFLoader(str(pdf_file))

            documents = loader.load()

            # Add metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)

            print(f"Successfully processed: {len(documents)} pages.")

        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")

    print(f"\nTotal documents processed: {len(all_documents)}")

    return all_documents


# Process all PDFs
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files in the directory '../data'.

Processing file: ..\data\pdf\Front End Developer - Preparation Document-1 (1).pdf
Successfully processed: 3 pages.

Processing file: ..\data\pdf\Software Developer - Preparation Document.pdf
Successfully processed: 3 pages.

Total documents processed: 6


In [20]:

## Text Splitting into Chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """
    Splits documents into smaller chunks using
    RecursiveCharacterTextSplitter.
    """

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    all_chunks = []

    for doc in documents:

        # Split document text
        chunks = text_splitter.split_text(doc.page_content)

        # Create Document chunk objects
        for i, chunk in enumerate(chunks):

            chunk_doc = Document(
                page_content=chunk,
                metadata={
                    **doc.metadata,
                    "chunk_index": i
                }
            )

            all_chunks.append(chunk_doc)

    print(f"Total chunks created: {len(all_chunks)}")

    return all_chunks

In [19]:
chunks=split_documents(all_pdf_documents)
chunks

Total chunks created: 14


[Document(metadata={'producer': 'Skia/PDF m102 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\Front End Developer - Preparation Document-1 (1).pdf', 'file_path': '..\\data\\pdf\\Front End Developer - Preparation Document-1 (1).pdf', 'total_pages': 3, 'format': 'PDF 1.4', 'title': 'Front End Developer - Preparation Document', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'Front End Developer - Preparation Document-1 (1).pdf', 'file_type': 'pdf', 'chunk_index': 0}, page_content="Front End Developer - Preparation Document\nDear Students,\nWe hire tech enthusiasts with a broad set of technical skills who are ready to tackle\nsome of technology's greatest challenges. The hiring process has been designed from\nthe ground level to avoid any false positives and in order to help you to get through\nour process.\nWe have curated this document after examining some of the 

### Embedding and VectorStoreDB

In [21]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity



d:\Study Material\VS Studio\Machine Learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
class EmbeddingManager:
    """handles text embedding generation using SentenceTransformer models."""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initializes the embedding manager with a specified SentenceTransformer model
        
        args:
            model_name : huggingface model name for sentence embedding
        
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Loads the SentenceTransformer model."""
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Successfully loaded embedding model: {self.model_name}")
            print(f"model loaded successfully . embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise e

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generates embeddings for a list of texts.
        
        args:
            texts : list of input texts to generate embeddings for
        returns:
            numpy array of shape (len(texts), embedding_dimension) containing the generated embeddings
        """
        if not self.model:
            raise ValueError("Model not loaded. Call _load_model() first.")
        try:
            embeddings = self.model.encode(texts, show_progress_bar=True)
            return np.array(embeddings)
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise e


## Initializing the embedding manager and generating embeddings for the chunks
embedding_manager = EmbeddingManager()
embedding_manager



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3071.11it/s]


Successfully loaded embedding model: all-MiniLM-L6-v2
model loaded successfully . embedding dimension: 384


C:\Users\sachi\AppData\Local\Temp\ipykernel_13168\2666584305.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"model loaded successfully . embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [25]:
### Vector Database Management using ChromaDB

import uuid
from typing import List, Any

import chromadb
import numpy as np

from chromadb.config import Settings


class VectorStore:
    """
    A simple vector store implementation using ChromaDB
    for storing and retrieving document embeddings.
    """

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "./chroma_db"
    ):
        """
        Initializes the vector store.

        Args:
            collection_name:
                Name of the ChromaDB collection

            persist_directory:
                Directory where the ChromaDB database
                will be persisted
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory

        self.client = None
        self.collection = None

        self._initialize_chromadb()

    def _initialize_chromadb(self):
        """Initializes the ChromaDB client and collection."""

        try:

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            self.collection = (
                self.client.get_or_create_collection(
                    name=self.collection_name
                )
            )

            print(
                f"ChromaDB initialized with "
                f"collection: {self.collection_name}"
            )

        except Exception as e:

            print(f"Error initializing ChromaDB: {e}")
            raise e

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """
        Adds documents and embeddings to vector store.

        Args:
            documents:
                List of document objects

            embeddings:
                Numpy array of embeddings
        """

        if len(documents) != len(embeddings):

            raise ValueError(
                "The number of documents and embeddings "
                "must be the same."
            )

        print(
            f"Adding {len(documents)} documents "
            f"to the vector store..."
        )

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            # Generate unique ID
            doc_id = f"{uuid.uuid4().hex[:8]}_{i}"

            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)

            metadata["doc_index"] = i

            metadatas.append(metadata)

            # Store content + embedding
            documents_text.append(doc.page_content)

            embeddings_list.append(
                embedding.tolist()
            )

        # Add to ChromaDB
        try:

            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )

            print(
                f"Successfully added "
                f"{len(documents)} documents "
                f"to the vector store."
            )

        except Exception as e:

            print(
                f"Error adding documents "
                f"to vector store: {e}"
            )

            raise e


# Initialize Vector Store
vectorstore = VectorStore()

print(vectorstore)

ChromaDB initialized with collection: pdf_documents


In [26]:
chunks

[Document(metadata={'producer': 'Skia/PDF m102 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\Front End Developer - Preparation Document-1 (1).pdf', 'file_path': '..\\data\\pdf\\Front End Developer - Preparation Document-1 (1).pdf', 'total_pages': 3, 'format': 'PDF 1.4', 'title': 'Front End Developer - Preparation Document', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'Front End Developer - Preparation Document-1 (1).pdf', 'file_type': 'pdf', 'chunk_index': 0}, page_content="Front End Developer - Preparation Document\nDear Students,\nWe hire tech enthusiasts with a broad set of technical skills who are ready to tackle\nsome of technology's greatest challenges. The hiring process has been designed from\nthe ground level to avoid any false positives and in order to help you to get through\nour process.\nWe have curated this document after examining some of the 

In [33]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

## Add generate_embeddings to the current kernel object if it was created before the class was fixed
if not hasattr(embedding_manager, "generate_embeddings"):
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded. Call _load_model() first.")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        return np.array(embeddings)

    EmbeddingManager.generate_embeddings = generate_embeddings

## Generate the embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

## Store in the vector database
vectorstore.add_documents(chunks, embeddings)

Batches: 100%|██████████| 1/1 [00:02<00:00,  2.53s/it]


Adding 14 documents to the vector store...
Successfully added 14 documents to the vector store.


### RAG Retriever From VextorStore

In [40]:
class RAGRetriever:
    """handles query-based retrieval from the vector store."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Initializes the retriever with a vector store and embedding manager.
        
        Args:
            vector_store : instance of VectorStore for retrieval
            embedding_manager : instance of EmbeddingManager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        """Retrieves relevant documents from the vector store based on the query.
        
        Args:
            query : input query string for retrieval
            top_k : number of top relevant documents to retrieve

        Returns:
            List[Dict[str, Any]]: List of retrieved documents with their scores.
        """
        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #search in the vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                include=["metadatas", "documents", "embeddings"]
            )

            # Process results
            retrieved_docs = []
            for doc, metadata, embedding in zip(
                results["documents"][0],
                results["metadatas"][0],
                results["embeddings"][0]
            ):
                score = cosine_similarity(
                    [query_embedding],
                    [embedding]
                )[0][0]

                retrieved_docs.append({
                    "document": doc,
                    "metadata": metadata,
                    "score": score
                })

            # Sort by score
            retrieved_docs.sort(key=lambda x: x["score"], reverse=True)

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            raise e

# Initialize RAG Retriever
rag_retriever = RAGRetriever(vector_store=vectorstore, embedding_manager=embedding_manager)
rag_retriever.retrieve("What is the main topic of the document?")

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.54it/s]


[{'document': "Software Developer - Preparation Document\nDear Students,\nWe hire tech enthusiasts with a broad set of technical skills who are ready to tackle\nsome of technology's greatest challenges. The hiring process has been designed from\nthe ground level to avoid any false positives and in order to help you to get through\nour process.\nWe have curated this document after examining some of the most frequently asked\nquestions & also keeping in mind the preparation that you may require in order to\ncrack our selection process. This document will come in handy in order to understand\nthe position and the tips and tricks that will help you prepare for the hiring process for\nSoftware Developer at Josh Technology Group.\nPlease Note:- This document is intended to provide you with the required guidance\nand sample material that would be helpful in the preparation and this in no way\nguarantees your selection.\nExcited much to participate in the selection process? We look forward to 

### Integration Vectordb Context pipeline With LLM output

In [6]:
## Simple RAG Pipeline with groq LLM
from langchain_groq import ChatGroq
import os
from pathlib import Path
from typing import Any, Dict, List

import chromadb
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
load_dotenv()

# 1. Initialize the groq LLM(set up API key in .env file)
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY is not set. Add it to your .env file.")

groq_model = os.getenv("GROQ_MODEL", "llama-3.1-8b-instant")
llm = ChatGroq(groq_api_key=groq_api_key, model=groq_model, temperature=0.1, max_tokens=1024)

class ChromaRetrieverFallback:
    """Small retriever that can be used after a kernel restart."""

    def __init__(self, collection_name: str = "pdf_documents", model_name: str = "all-MiniLM-L6-v2"):
        possible_paths = [Path("chroma_db"), Path("notebook/chroma_db")]
        persist_path = next((path for path in possible_paths if path.exists()), None)
        if persist_path is None:
            raise RuntimeError("ChromaDB was not found. Run the vector store setup cells first.")

        self.client = chromadb.PersistentClient(path=str(persist_path))
        self.collection = self.client.get_collection(name=collection_name)
        self.model = SentenceTransformer(model_name)

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        query_embedding = self.model.encode([query])[0]
        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k,
            include=["metadatas", "documents", "distances"],
        )

        retrieved_docs = []
        for doc, metadata, distance in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        ):
            retrieved_docs.append({
                "document": doc,
                "metadata": metadata,
                "score": 1 / (1 + distance),
            })

        return retrieved_docs


if "rag_retriever" not in globals():
    if all(name in globals() for name in ["RAGRetriever", "vectorstore", "embedding_manager"]):
        rag_retriever = RAGRetriever(vector_store=vectorstore, embedding_manager=embedding_manager)
    else:
        rag_retriever = ChromaRetrieverFallback()

# 2. Retrieve relevant documents using RAGRetriever
def retrieve_relevant_docs(query: str, top_k: int = 3) -> List[Dict[str, Any]]:
    """Retrieves relevant documents for a given query using RAGRetriever."""
    return rag_retriever.retrieve(query, top_k=top_k)

# 3. Generate response using the retrieved documents and the LLM
def generate_response(query: str) -> str:
    """Generates a response for the given query using retrieved documents and the LLM."""
    retrieved_docs = retrieve_relevant_docs(query)

    # Combine retrieved documents into a single context string
    context = "\n\n".join(
        [f"Document: {doc['document']}\nMetadata: {doc['metadata']}" for doc in retrieved_docs]
    )

    # Create a prompt for the LLM
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"

    # Generate response from the LLM
    response = llm.invoke(prompt)

    return response.content

# Example usage
query = "What is the specific things josh looking for in a candidate?"
response = generate_response(query)
print("Response from LLM:")
print(response)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1075.39it/s]


Response from LLM:
Based on the provided documents, Josh Technology Group is looking for tech enthusiasts with a broad set of technical skills who are ready to tackle some of technology's greatest challenges. However, the specific things Josh is looking for in a candidate are not explicitly mentioned in the provided documents.

But, we can infer that Josh is looking for candidates who:

1. Have a broad set of technical skills.
2. Are ready to tackle some of technology's greatest challenges.
3. Can crack their selection process, which implies that they are looking for candidates who are prepared and have a good understanding of the position and the hiring process.

It would be helpful to review the actual hiring process and selection criteria to get a more accurate understanding of what Josh is looking for in a candidate.


### Enhance RAG Pipeline

In [7]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """Advanced RAG function that retrieves relevant documents and generates a response using the LLM.
    
    Args:
        query : input query string
        retriever : instance of RAGRetriever for retrieval
        llm : instance of ChatGroq for response generation
        top_k : number of top relevant documents to retrieve
        min_score : minimum cosine similarity score for retrieved documents to be considered relevant
        return_context : whether to return the retrieved context along with the response
    Returns:
        str or Tuple[str, List[Dict[str, Any]]]: Generated response from the LLM, optionally with retrieved context
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return "No relevant documents found." if not return_context else ("No relevant documents found.", [])

    context = "\n\n".join(doc["document"] for doc in results)
    sources =[{
        'source': doc['metadata']['source_file'],
        'page': doc['metadata'].get('source_page', 'N/A'),
        'score': doc['score'],
        'previous': doc['metadata'].get('previous_chunk', 'N/A')
    }]
    for doc in results:
        print(f"Source: {doc['metadata']['source_file']}, Page: {doc['metadata'].get('source_page', 'N/A')}, Score: {doc['score']:.4f}, Previous Chunk: {doc['metadata'].get('previous_chunk', 'N/A')}")    
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    response = llm.invoke(prompt)

    if return_context:
        return response.content, sources
    else:
        return response.content


generate_response("What is the specific things josh looking for in a candidate?")


"Based on the provided documents, it seems that Josh Technology Group is looking for tech enthusiasts with a broad set of technical skills who are ready to tackle some of technology's greatest challenges. However, the specific things they are looking for in a candidate are not explicitly mentioned in the provided documents.\n\nBut, we can infer some key skills and qualities that Josh Technology Group might be looking for in a candidate based on the context and the fact that they have created preparation documents for Front End Developer and Software Developer positions.\n\nSome of the key skills and qualities that Josh Technology Group might be looking for in a candidate include:\n\n1. Technical skills: A broad set of technical skills, including programming languages, software development methodologies, and tools.\n2. Problem-solving skills: The ability to tackle complex technical challenges and solve problems.\n3. Communication skills: The ability to communicate technical ideas and so